# 🏦🌸 Sistema de Análisis de Riesgo Crediticio
### Sistema Financiero Colombiano — Datos Abiertos Gov.co
**Autores:** Angela Rico · Sebastian Ramirez  
**Curso:** Python para APIs e IA Aplicada — Universidad Santo Tomás · 2026  
**Fecha:** febrero 28, 2026

---

## 📋 Conceptos Aplicados en este Notebook

| Semana | Concepto | Dónde se usa |
|--------|----------|---------------|
| 1 | **Pattern Matching** (`match/case`) | Clasificación de riesgo NPL |
| 1 | **Decoradores simples** | `@registrar_ejecucion` en funciones de análisis |
| 1 | **Decorator Factory** | `@validar_normalidad(alpha=0.05)` |
| 1 | **Type Hints** (`Literal`, `Optional`) | Todo el código |
| 2 | **Pydantic** (`BaseModel`, `Field`) | Validación de datos de la API |
| 2 | **OOP — Herencia** | `AnalizadorMuestral` hereda de `AnalizadorEstadistico` |
| 2 | **OOP — Polimorfismo** | `calcular_varianza()` con ddof=0 vs ddof=1 |
| 2 | **HTTP + `requests.Session`** | Cliente API con conexión persistente |
| 3 | **FastAPI** | Endpoints CRUD con Swagger UI |
| 3 | **Visualizaciones** | Paleta coherente "rosadita" 🌸 |


---
## 0. 🔧 Setup — Configuración del Entorno

> **CONCEPTO: Entornos Virtuales (Semana 1)**  
> Usamos `venv` + `requirements.txt` para que cualquier persona reproduzca 
> exactamente el mismo entorno. Sin esto, "funciona en mi máquina" es inevitable.


In [ ]:
# Imports del proyecto
import sys
import numpy as np
import pandas as pd
from datetime import datetime

# Módulos locales del proyecto
from config import PALETA, RIESGO_COLOR, RUTA_SALIDA, UMBRALES_RIESGO, ALPHA_DEFAULT
from modelos import (
    MunicipioFinanciero, ResultadoAnalisis, VariableEstadistica,
    clasificar_riesgo, validar_dataframe, NivelRiesgo,
    AnalizadorEstadistico, AnalizadorPoblacional, AnalizadorMuestral,  # OOP
)
from decorators import registrar_ejecucion, validar_normalidad, muestra_minima

print(f"Python {sys.version}")
print(f"NumPy  {np.__version__}")
print(f"Pandas {pd.__version__}")
print("✅ Todos los módulos importados correctamente")


---
## 1. 🛡️ Validación con Pydantic (Semana 2)

> **CONCEPTO: Pydantic BaseModel + Field**  
> Pydantic valida datos **automáticamente** al crear objetos.  
> A diferencia de las anotaciones normales de Python (que son opcionales),  
> Pydantic **lanza ValidationError inmediatamente** si los datos no cumplen el contrato.  
>  
> **¿Por qué Pydantic y no solo `isinstance()`?**  
> 1. Validación automática + conversión de tipos ("1,500" → 1500.0)  
> 2. Mensajes de error claros para el usuario  
> 3. Integración directa con FastAPI (Swagger UI documenta el schema)  


In [ ]:
# ── Demo 1: VariableEstadistica — campos con restricciones ──────────────
print("═" * 60)
print("  DEMO: Validación Pydantic — VariableEstadistica")
print("═" * 60)

# Caso válido (con normalización automática del nombre)
v1 = VariableEstadistica(
    nombre="  índice de mora  ",  # Pydantic normaliza → "Índice De Mora"
    desviacion_std=0.015,         # Field(gt=0): debe ser > 0
    probabilidad=0.27,            # Field(ge=0, le=1): entre 0 y 1
    valor_maximo=0.85,            # Field(ge=0): opcional, >= 0
)
print(f"\n✅ Nombre normalizado: '{v1.nombre}'")
print(f"   Probabilidad: {v1.probabilidad}")
print(f"   Unidad (default): {v1.unidad}")


In [ ]:
# Caso inválido: desviación estándar negativa
from pydantic import ValidationError

print("\n❌ Caso inválido — desviación negativa:")
try:
    VariableEstadistica(nombre="mora", desviacion_std=-0.5, probabilidad=0.3)
except ValidationError as e:
    print(f"   Campo: {e.errors()[0]['loc'][0]}")
    print(f"   Error: {e.errors()[0]['msg']}")
    print("   → Pydantic RECHAZA el objeto antes de crearlo")


In [ ]:
# ── Demo 2: MunicipioFinanciero — datos sucios de la API ────────────────
print("\n" + "═" * 60)
print("  DEMO: Coerción automática — datos sucios → limpios")
print("═" * 60)

# La API puede devolver números como strings con comas/puntos
muni = MunicipioFinanciero(
    municipio="  bogotá d.c.  ",          # → "Bogotá D.C."
    cartera_a="1,500,000,000",            # → 1500000000.0
    cartera_c="50.000.000",               # → 50000000.0
    cartera_d="25.000.000",               # → 25000000.0
    cartera_e="10.000.000",               # → 10000000.0
    total_cartera="1,600,000,000",        # → 1600000000.0
    total_captaciones="2.000.000.000",    # → 2000000000.0
).calcular_indicadores()

print(f"\n  municipio:      {muni.municipio}")
print(f"  cartera_a:      {muni.cartera_a:,.0f} COP")
print(f"  total_cartera:  {muni.total_cartera:,.0f} COP")
print(f"  indice_riesgo:  {muni.indice_riesgo:.4%}")
print(f"  ratio_liquidez: {muni.ratio_liquidez:.3f}")
print(f"  nivel_riesgo:   {muni.nivel_riesgo}")


---
## 2. 🏗️ OOP — Herencia y Polimorfismo (Semana 2)

> **CONCEPTO: Herencia**  
> `AnalizadorPoblacional` y `AnalizadorMuestral` **heredan** de `AnalizadorEstadistico`.  
> Heredan TODOS los métodos (media, mediana, resumen) sin re-escribirlos.
>
> **CONCEPTO: Polimorfismo**  
> El método `calcular_varianza()` tiene la **misma firma** en ambas clases,  
> pero el **comportamiento cambia**:  
> - Poblacional: `ddof=0` → divide entre N  
> - Muestral: `ddof=1` → divide entre N-1 (corrección de Bessel)  
>
> **Analogía con clase (Semana II):**  
> Igual que `Moneda.__init__(valor)` vs `MonedaExtranjera.__init__(valor, cambio)`,  
> pero aplicado a varianza poblacional vs muestral.


In [ ]:
# ── Demo: Polimorfismo — mismos datos, diferente resultado ──────────────
print("═" * 60)
print("  DEMO: Polimorfismo — Poblacional vs Muestral")
print("═" * 60)

# Datos de ejemplo: cartera de 5 municipios (millones COP)
datos_cartera = [1500, 200, 50, 25, 10]

# MISMO constructor, MISMOS datos → DIFERENTE varianza
poblacional = AnalizadorPoblacional("Cartera (población)", datos_cartera)
muestral    = AnalizadorMuestral("Cartera (muestra)", datos_cartera)

res_pob = poblacional.resumen()
res_mus = muestral.resumen()

print(f"\n  {'Estadístico':<20} {'Poblacional':>15} {'Muestral':>15}")
print(f"  {'─'*20} {'─'*15} {'─'*15}")
print(f"  {'Tipo':<20} {res_pob['tipo']:>15} {res_mus['tipo']:>15}")
print(f"  {'Media':<20} {res_pob['media']:>15,.2f} {res_mus['media']:>15,.2f}")
print(f"  {'Mediana':<20} {res_pob['mediana']:>15,.2f} {res_mus['mediana']:>15,.2f}")
print(f"  {'Varianza':<20} {res_pob['varianza']:>15,.2f} {res_mus['varianza']:>15,.2f}  ← DIFERENTE")
print(f"  {'Std':<20} {res_pob['std']:>15,.2f} {res_mus['std']:>15,.2f}  ← DIFERENTE")
print(f"  {'CV (%)':<20} {res_pob['cv_pct']:>15,.2f} {res_mus['cv_pct']:>15,.2f}")

print(f"\n  📊 Varianza poblacional: σ² = Σ(xᵢ-μ)² / N     = {res_pob['varianza']:,.2f}")
print(f"  📊 Varianza muestral:    s² = Σ(xᵢ-x̄)² / (N-1) = {res_mus['varianza']:,.2f}")
print(f"\n  → La muestral es MAYOR porque divide entre N-1 (corrección de Bessel)")
print(f"  → Esto compensa la subestimación al usar una muestra")


---
## 3. 🎀 Decoradores (Semana 1)

> **CONCEPTO: Decorador Simple**  
> `@registrar_ejecucion` mide tiempo sin modificar el código de la función.  
> Es como ponerle un "cronómetro" a cualquier función con una sola línea.
>
> **CONCEPTO: Decorator Factory**  
> `@validar_normalidad(alpha=0.05)` recibe un parámetro.  
> Es una función que RETORNA un decorador (función que retorna función).
>
> **¿Por qué decoradores y no un print() directo?**  
> 1. **Reutilizable**: se aplica a cualquier función  
> 2. **Separación de responsabilidades**: la función no sabe que está siendo medida  
> 3. **Open/Closed Principle**: se puede quitar/poner sin tocar la lógica  


In [ ]:
# ── Demo: Decorador simple @registrar_ejecucion ─────────────────────────
print("═" * 60)
print("  DEMO: @registrar_ejecucion — Decorador Simple")
print("═" * 60)

@registrar_ejecucion
def calcular_estadisticos_demo(data):
    """Función decorada — el decorador mide su tiempo."""
    arr = np.array(data, dtype=float)
    return {
        "media": float(np.mean(arr)),
        "mediana": float(np.median(arr)),
        "std": float(np.std(arr, ddof=1)),
    }

# Al llamar, el decorador imprime timestamps automáticamente
resultado = calcular_estadisticos_demo([100, 200, 300, 400, 500])
print(f"\n  Resultado: {resultado}")


In [ ]:
# ── Demo: Decorator Factory @validar_normalidad(alpha) ──────────────────
print("\n" + "═" * 60)
print("  DEMO: @validar_normalidad(alpha=0.05) — Decorator Factory")
print("═" * 60)

@validar_normalidad(alpha=0.05)  # Factory: recibe parámetro
def analizar_distribucion(data):
    """Analiza datos — el decorador valida normalidad ANTES."""
    return {"n": len(data), "media": float(np.mean(data))}

# Datos no normales (exponencial)
datos_no_normales = list(np.random.exponential(scale=100, size=50))
resultado = analizar_distribucion(datos_no_normales)
print(f"\n  Resultado: {resultado}")


---
## 4. 🔀 Pattern Matching (Semana 1)

> **CONCEPTO: match/case (Python 3.10+)**  
> Más expresivo que `if-elif-else` para clasificar datos.  
> El intérprete puede optimizarlo mejor y escala limpiamente.


In [ ]:
# ── Demo: Pattern Matching para clasificación de riesgo ──────────────────
print("═" * 60)
print("  DEMO: Pattern Matching — Clasificación NPL")
print("═" * 60)

# Umbrales: Basel II/III + Circular 98 Superfinanciera
casos_test = [
    {"indice_riesgo": 0.0},     # sin_riesgo
    {"indice_riesgo": 0.005},   # riesgo_bajo
    {"indice_riesgo": 0.03},    # riesgo_moderado
    {"indice_riesgo": 0.10},    # riesgo_alto
    {"indice_riesgo": 0.25},    # riesgo_critico
    {},                          # sin_datos
]

print(f"\n  {'NPL Ratio':<15} {'Clasificación':<20} {'Color'}")
print(f"  {'─'*15} {'─'*20} {'─'*15}")
for caso in casos_test:
    nivel = clasificar_riesgo(caso)
    color = RIESGO_COLOR.get(nivel, '#999')
    idx = caso.get('indice_riesgo', 'N/A')
    print(f"  {str(idx):<15} {nivel:<20} {color}")


---
## 5. 📐 Selección Automática de Prueba Estadística

> **CONCEPTO: Plantilla inteligente**  
> El sistema examina los datos (tamaño de muestra, distribución)  
> y automáticamente elige la prueba de normalidad más apropiada:  
>
> | Tamaño (n) | Prueba seleccionada | Justificación |
> |-----------|---------------------|---------------|
> | n < 50 | **Shapiro-Wilk** | Óptima para muestras pequeñas |
> | 50 ≤ n ≤ 5000 | **Shapiro-Wilk + K-S** | Comparar ambas |
> | n > 5000 | **Anderson-Darling** | Diseñada para muestras grandes |


In [ ]:
from scipy import stats

def seleccionar_prueba_normalidad(datos, alpha=0.05):
    """
    Selecciona automáticamente la prueba de normalidad según n.
    
    JUSTIFICACIÓN:
      - Shapiro-Wilk: más potente para n < 50 (Razali & Wah, 2011)
      - K-S (Lilliefors): complemento para n medianos
      - Anderson-Darling: robusta para n grandes, enfatiza colas
    """
    n = len(datos)
    resultados = {"n": n, "alpha": alpha, "pruebas": []}
    
    if n < 8:
        print(f"  ⚠ n={n} es muy pequeño para tests de normalidad")
        resultados["conclusion"] = "muestra_insuficiente"
        return resultados
    
    # Selección por tamaño de muestra
    match n:
        case n if n < 50:
            print(f"  📊 n={n} < 50 → Shapiro-Wilk (óptima para muestras pequeñas)")
            stat, p = stats.shapiro(datos)
            resultados["pruebas"].append({"nombre": "Shapiro-Wilk", "stat": round(stat,4), "p": round(p,4)})
        case n if n <= 5000:
            print(f"  📊 50 ≤ n={n} ≤ 5000 → Shapiro-Wilk + Kolmogorov-Smirnov")
            stat_sw, p_sw = stats.shapiro(datos)
            stat_ks, p_ks = stats.kstest(datos, 'norm', args=(np.mean(datos), np.std(datos)))
            resultados["pruebas"].append({"nombre": "Shapiro-Wilk", "stat": round(stat_sw,4), "p": round(p_sw,4)})
            resultados["pruebas"].append({"nombre": "K-S (Lilliefors)", "stat": round(stat_ks,4), "p": round(p_ks,4)})
        case _:
            print(f"  📊 n={n} > 5000 → Anderson-Darling (robusta para muestras grandes)")
            result = stats.anderson(datos, dist='norm')
            resultados["pruebas"].append({"nombre": "Anderson-Darling", "stat": round(result.statistic,4)})
    
    # Conclusión basada en p-values
    for prueba in resultados["pruebas"]:
        if "p" in prueba:
            es_normal = prueba["p"] > alpha
            prueba["es_normal"] = es_normal
            emoji = "✅" if es_normal else "❌"
            print(f"  {emoji} {prueba['nombre']}: p={prueba['p']:.4f} {'>' if es_normal else '≤'} α={alpha}")
            print(f"     → {'NO se rechaza H₀ (compatible con normalidad)' if es_normal else 'Se RECHAZA H₀ (NO es normal)'}")
    
    return resultados

# Demo con datos simulados
print("═" * 60)
print("  DEMO: Selección automática de prueba")
print("═" * 60)

np.random.seed(42)
datos_demo = list(np.random.exponential(scale=100, size=200))
resultado_test = seleccionar_prueba_normalidad(datos_demo)


---
## 6. 🌸 Visualizaciones — Estilo "Rosadito Bonito"

> La paleta coherente se configura en `config.py` y se hereda
> automáticamente en TODAS las gráficas vía `plt.rcParams`.


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# ── Gráfica 1: Distribución de riesgo por nivel ─────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('🌸 Panel de Análisis — Estilo Rosadito', fontsize=14, fontweight='bold',
             color=PALETA['oscuro'], y=1.02)

# Subplot 1: Barras de niveles de riesgo
niveles = list(RIESGO_COLOR.keys())
valores = [35, 25, 20, 12, 5, 3]  # Ejemplo: distribución de municipios
colores = [RIESGO_COLOR[n] for n in niveles]

axes[0].barh(niveles, valores, color=colores, edgecolor='white', linewidth=1.5)
axes[0].set_title('Municipios por Nivel de Riesgo', color=PALETA['oscuro'])
axes[0].set_xlabel('Cantidad de municipios')
for i, v in enumerate(valores):
    axes[0].text(v + 0.5, i, f'{v}', va='center', fontweight='bold', fontsize=9)

# Subplot 2: Composición de cartera (pie)
categorias = ['A (Normal)', 'B (Observación)', 'C (Subestándar)', 'D (Dudosa)', 'E (Pérdida)']
pcts = [75, 10, 8, 4, 3]
colores_pie = [PALETA['sin_riesgo'], PALETA['riesgo_bajo'], PALETA['riesgo_mod'],
               PALETA['riesgo_alto'], PALETA['riesgo_crit']]
wedges, texts, autotexts = axes[1].pie(
    pcts, labels=categorias, colors=colores_pie, autopct='%1.1f%%',
    startangle=90, pctdistance=0.85,
    wedgeprops={'edgecolor': 'white', 'linewidth': 2}
)
for t in autotexts:
    t.set_fontsize(8)
    t.set_fontweight('bold')
axes[1].set_title('Composición de Cartera (Clasificación Regulatoria)', color=PALETA['oscuro'])

plt.tight_layout()
plt.savefig(RUTA_SALIDA / 'panel_rosadito.png', dpi=300, bbox_inches='tight',
            facecolor='white', edgecolor='none')
plt.show()
print(f"\n✅ Gráfica guardada en: {RUTA_SALIDA / 'panel_rosadito.png'}")


In [ ]:
# ── Gráfica 2: Histograma con estilo rosadito ───────────────────────────
np.random.seed(42)
indices_riesgo = np.random.exponential(scale=0.03, size=200)
indices_riesgo = np.clip(indices_riesgo, 0, 1)

fig, ax = plt.subplots(figsize=(10, 5))

# Histograma con color rosadito
n_bins, bins, patches = ax.hist(
    indices_riesgo, bins=30, color=PALETA['primario'],
    alpha=0.7, edgecolor=PALETA['secundario'], linewidth=0.8
)

# Líneas de media y mediana
media = np.mean(indices_riesgo)
mediana = np.median(indices_riesgo)
ax.axvline(media, color=PALETA['secundario'], linestyle='--', linewidth=2,
           label=f'Media = {media:.4f}')
ax.axvline(mediana, color=PALETA['acento'], linestyle='-', linewidth=2,
           label=f'Mediana = {mediana:.4f}')

# Zonas de riesgo como fondo
ax.axvspan(0, 0.01, alpha=0.1, color=PALETA['sin_riesgo'], label='Riesgo bajo')
ax.axvspan(0.01, 0.05, alpha=0.1, color=PALETA['riesgo_mod'], label='Moderado')
ax.axvspan(0.05, 0.15, alpha=0.1, color=PALETA['riesgo_alto'], label='Alto')
ax.axvspan(0.15, 1.0, alpha=0.1, color=PALETA['riesgo_crit'], label='Crítico')

ax.set_title('🌸 Distribución del Índice NPL — Media vs Mediana',
             fontsize=13, color=PALETA['oscuro'])
ax.set_xlabel('Índice de Riesgo NPL')
ax.set_ylabel('Frecuencia')
ax.legend(loc='upper right', fontsize=8)

# Interpretación
diferencia = abs(media - mediana)
interpretacion = 'asimetría positiva (cola derecha)' if media > mediana else 'distribución simétrica'
ax.text(0.98, 0.85, f'Media - Mediana = {diferencia:.4f}\n→ {interpretacion}',
        transform=ax.transAxes, ha='right', va='top', fontsize=8,
        bbox=dict(boxstyle='round,pad=0.5', facecolor=PALETA['terciario'], alpha=0.8))

plt.tight_layout()
plt.savefig(RUTA_SALIDA / 'histograma_rosadito.png', dpi=300, bbox_inches='tight',
            facecolor='white')
plt.show()
print(f"✅ Gráfica guardada en: {RUTA_SALIDA / 'histograma_rosadito.png'}")


---
## 7. 🌐 API FastAPI (Semana 3)

> **CONCEPTO: FastAPI decorators** (`@app.get`, `@app.post`, `@app.delete`)  
> Cada decorador **mapea una URL a una función Python**.
>
> **CONCEPTO: CRUD** — Create (POST), Read (GET), Update (PUT), Delete (DELETE)  
>
> **Para probar la API:**
> ```bash
> uvicorn main:app --reload --host 0.0.0.0 --port 8000
> ```
> Luego visite: **http://127.0.0.1:8000/docs**
>
> | Verbo | Ruta | Descripción | Status |
> |-------|------|-------------|--------|
> | GET | `/` | Health check | 200 |
> | POST | `/analizar` | Crear análisis | 201/422 |
> | GET | `/historial` | Listar análisis | 200 |
> | GET | `/historial/{id}` | Obtener análisis | 200/404 |
> | DELETE | `/historial/{id}` | Eliminar análisis | 200/404 |


In [ ]:
# Demo de la función pura (independiente de FastAPI)
from main import procesar_riesgo_crediticio

print("═" * 60)
print("  DEMO: procesar_riesgo_crediticio() — Función pura")
print("═" * 60)

datos_test = {
    "cartera_a": 1500000000,
    "cartera_b": 200000000,
    "cartera_c": 50000000,
    "cartera_d": 25000000,
    "cartera_e": 10000000,
    "total_cartera": 1800000000,
    "total_captaciones": 2500000000,
}

resultado = procesar_riesgo_crediticio(datos_test)

print(f"\n  Índice NPL:         {resultado['indice_riesgo']:.4%}")
print(f"  Nivel de riesgo:    {resultado['nivel_riesgo']}")
print(f"  Mensaje:            {resultado['mensaje']}")
print(f"  Tipo análisis:      {resultado['tipo_analisis']} (ddof=1)")
print(f"  Cartera sana:       {resultado['pct_cartera_sana']:.2f}%")
print(f"  Cartera en mora:    {resultado['pct_cartera_mora']:.2f}%")


---
## 8. ✅ Conclusiones

### Resumen de Conceptos Aplicados

| # | Concepto | Archivo | Ejemplo concreto |
|---|---------|---------|------------------|
| 1 | Pattern Matching | `modelos.py`, `main.py` | `match idx: case 0.0: ...` |
| 2 | Decorador simple | `main.py` | `@registrar_ejecucion` |
| 3 | Decorator Factory | `decorators.py` | `@validar_normalidad(alpha=0.05)` |
| 4 | Type Hints | Todos | `Literal["sin_riesgo", ...]` |
| 5 | Pydantic BaseModel | `modelos.py` | `MunicipioFinanciero(BaseModel)` |
| 6 | Pydantic Field | `modelos.py` | `Field(ge=0, min_length=2)` |
| 7 | field_validator | `modelos.py` | `limpiar_numeros(mode="before")` |
| 8 | OOP — Herencia | `modelos.py` | `AnalizadorMuestral(AnalizadorEstadistico)` |
| 9 | OOP — Polimorfismo | `modelos.py` | `calcular_varianza()` ddof=0 vs ddof=1 |
| 10 | OOP — Constructor | `api_client.py` | `__init__(self, base_url, timeout)` |
| 11 | requests.Session | `api_client.py` | Conexión TCP persistente |
| 12 | FastAPI CRUD | `main.py` | POST, GET, DELETE endpoints |
| 13 | Swagger UI | `main.py` | Documentación automática en `/docs` |
| 14 | NumPy | `main.py` | 12 cálculos estadísticos |
| 15 | Entorno virtual | `requirements.txt` | Reproducibilidad total |

### Interpretación Estadística

- El **índice NPL** (Non-Performing Loans) mide la proporción de cartera en mora
- La **mediana** se prefiere sobre la media cuando hay asimetría (datos financieros)
- La **corrección de Bessel** (ddof=1) es necesaria porque trabajamos con muestras
- El **test de Shapiro-Wilk** confirma que los datos de cartera NO son normales

---
*Notebook generado como parte del Proyecto Personal — Semanas 1, 2 y 3*
